In [1]:
from flags import detect_scam_flags

msg = "URGENT! Your bank account is blocked. Send OTP immediately."

print(detect_scam_flags(msg))


{'otp_request': True, 'urgency': True, 'authority_impersonation': True, 'fear_threat': False, 'financial_pressure': False}


In [2]:
from risk_engine import compute_risk_score

ml_prob = 0.62
flags = {
    "otp_request": True,
    "urgency": True,
    "authority_impersonation": True,
    "fear_threat": False,
    "financial_pressure": False
}

print(compute_risk_score(ml_prob, flags))


{'risk_score': 62, 'risk_level': 'High'}


In [3]:
from explanation import generate_explanation

flags = {
    "otp_request": True,
    "urgency": True,
    "authority_impersonation": True,
    "fear_threat": False,
    "financial_pressure": False
}

print(generate_explanation(flags, "High"))


This message is classified as High risk because it requests sensitive information like an OTP, and creates urgency to pressure immediate action, and impersonates a trusted authority such as a bank or official service.


In [3]:
from pipeline import analyze_text

msg = "URGENT! Your bank account is blocked. Send OTP immediately."

print(analyze_text(msg))


ImportError: attempted relative import with no known parent package

In [5]:
import sys
sys.path.insert(0, '..')

from src.pipeline import analyze_text
import pandas as pd

# Real-world test cases
test_cases = [
    # LEGITIMATE MESSAGES (should be Low/Medium risk)
    {
        "message": "Hey, I'll be home at 6pm. See you soon!",
        "expected": "Low/Medium",
        "category": "Legitimate - Personal"
    },
    {
        "message": "Your package has been delivered. Thank you for shopping with us.",
        "expected": "Low",
        "category": "Legitimate - Order Notification"
    },
    {
        "message": "Meeting rescheduled to 3pm tomorrow. Please confirm.",
        "expected": "Low",
        "category": "Legitimate - Business"
    },
    {
        "message": "Hi! Your flight confirmation #AB1234 is attached in email.",
        "expected": "Low",
        "category": "Legitimate - Booking Confirmation"
    },
    
    # SCAM/SMISHING MESSAGES (should be High risk)
    {
        "message": "URGENT! Your bank account is blocked. Send OTP immediately.",
        "expected": "High",
        "category": "Scam - OTP Request + Urgency"
    },
    {
        "message": "You have won a free iPhone 14! Click www.claim-prize.com to claim now.",
        "expected": "High",
        "category": "Scam - Prize Claim"
    },
    {
        "message": "ALERT: Unusual login detected on your account. Verify at www.secure-verify.com ASAP",
        "expected": "High",
        "category": "Scam - Phishing"
    },
    {
        "message": "Congratulations! You are selected for KYC verification. Send photocopy of Aadhaar immediately.",
        "expected": "High",
        "category": "Scam - KYC Phishing"
    },
    {
        "message": "Your account will be suspended in 24 hours if you don't verify. Tap here to confirm.",
        "expected": "High",
        "category": "Scam - Fear Tactic"
    },
    {
        "message": "We tried to reach you! Your bank account needs immediate verification. Reply with account number.",
        "expected": "High",
        "category": "Scam - Bank Impersonation"
    },
    {
        "message": "WINNER! You are selected for Rs 5,00,000 prize draw. Transfer 500 rupees processing fee now.",
        "expected": "High",
        "category": "Scam - Financial Pressure"
    },
    {
        "message": "Dear customer, your airtime is running out. Confirm balance transfer to number 9876543210.",
        "expected": "High",
        "category": "Scam - Social Engineering"
    },
    {
        "message": "Final reminder: Your subscription renews tomorrow. Enter card details to continue service.",
        "expected": "High",
        "category": "Scam - Credential Harvesting"
    },
    {
        "message": "Police department official notice: You have pending traffic fine. Pay immediately or face legal action.",
        "expected": "High",
        "category": "Scam - Authority Impersonation + Fear"
    },
]

# Run tests and display results
results = []
print("=" * 100)
print(f"{'Category':<40} {'Message':<50} {'Risk Level':<12} {'Score':<8}")
print("=" * 100)

for test in test_cases:
    msg = test["message"]
    result = analyze_text(msg)
    
    results.append({
        "Category": test["category"],
        "Message": msg[:50] + "..." if len(msg) > 50 else msg,
        "Risk Level": result["risk_level"],
        "Risk Score": result["risk_score"],
        "Expected": test["expected"],
        "Full Message": msg,
        "Flags": result["flags"],
        "Explanation": result["explanation"]
    })
    
    print(f"{test['category']:<40} {msg[:40]:<50} {result['risk_level']:<12} {result['risk_score']:<8}")

print("=" * 100)
print("\n\nDETAILED RESULTS:")
print("=" * 100)

for i, res in enumerate(results, 1):
    print(f"\n{i}. {res['Category']}")
    print(f"   Message: {res['Full Message']}")
    print(f"   Risk Score: {res['Risk Score']}/100")
    print(f"   Risk Level: {res['Risk Level']}")
    print(f"   Flags: {res['Flags']}")
    print(f"   Explanation: {res['Explanation']}")
    print("-" * 100)

Category                                 Message                                            Risk Level   Score   
Legitimate - Personal                    Hey, I'll be home at 6pm. See you soon!            Low          1       
Legitimate - Order Notification          Your package has been delivered. Thank y           Medium       50      
Legitimate - Business                    Meeting rescheduled to 3pm tomorrow. Ple           Low          3       
Legitimate - Booking Confirmation        Hi! Your flight confirmation #AB1234 is            Low          16      
Scam - OTP Request + Urgency             URGENT! Your bank account is blocked. Se           High         79      
Scam - Prize Claim                       You have won a free iPhone 14! Click www           High         79      
Scam - Phishing                          ALERT: Unusual login detected on your ac           High         81      
Scam - KYC Phishing                      Congratulations! You are selected for KY       

In [6]:
# Summary Statistics
print("\n\n" + "=" * 100)
print("SUMMARY STATISTICS")
print("=" * 100)

legitimate_count = sum(1 for r in results if r['Expected'] == 'Low')
scam_count = sum(1 for r in results if r['Expected'] == 'High')

low_risk_correct = sum(1 for r in results if r['Expected'] == 'Low' and r['Risk Level'] == 'Low')
med_risk_correct = sum(1 for r in results if r['Expected'] == 'Low' and r['Risk Level'] == 'Medium')
high_risk_correct = sum(1 for r in results if r['Expected'] == 'High' and r['Risk Level'] == 'High')

print(f"\nLegitimate messages: {legitimate_count}")
print(f"  - Correctly classified as Low: {low_risk_correct}")
print(f"  - Incorrectly classified as Medium: {med_risk_correct}")

print(f"\nScam messages: {scam_count}")
print(f"  - Correctly classified as High: {high_risk_correct}")
print(f"  - Accuracy: {(high_risk_correct / scam_count * 100):.1f}%")

print(f"\nOverall Accuracy: {((low_risk_correct + high_risk_correct) / len(results) * 100):.1f}%")
print("=" * 100)



SUMMARY STATISTICS

Legitimate messages: 3
  - Correctly classified as Low: 2
  - Incorrectly classified as Medium: 1

Scam messages: 10
  - Correctly classified as High: 9
  - Accuracy: 90.0%

Overall Accuracy: 78.6%
